# The Spanish power system on one interactive map

A single self-contained HTML map carrying **every static attribute of the system** — no results,
no time dimension, just what the network and the fleet *are*. Saved to
`results/grid_maps/system_map_<SCENARIO>.html` and opened in your browser.
(For *operational* results — hourly flows, dispatch, voltages — use `results_plots_2035.ipynb`.)

Everything is precomputed into the file (~0.6 MB), so the control panel at the top right only
restyles layers that are already on the page. Nothing is recomputed and nothing is re-downloaded.

### Search
The box at the top of the panel does a case-insensitive substring match over **bus ids, line ids,
transformer ids, generating-unit ids and plant names** at once. Matches are highlighted with a
magenta halo drawn in a pane *below* the network, so a highlighted line still shows its own colour.

- Type to highlight live; the result list shows the first 25 with a type badge.
- **Click a result** to zoom to it and open its record.
- **Enter** zooms to fit every match — `LTGES04` lights up all 148 lines in that block.
- Paste a **comma- or space-separated list** of ids to highlight exactly that set.
- **Escape** clears.

A generating unit resolves to the bus it sits on: searching `Almaraz` finds the two nuclear units
and the co-located solar plant, and clicking one zooms to bus ES00158.

### Lines — recolour by any electrical attribute
Voltage level (discrete, the default) or a continuous scale over **thermal capacity**, **series
reactance**, **X:R ratio**, **circuits**, **length**, or **charging Mvar**. Continuous scales are
clipped to the 2nd–98th percentile so a few outliers don't flatten the ramp. Line **width always
follows thermal capacity**, and DC links are dashed, so those two stay readable in every mode.

A line popup gives the full record: voltage, circuits, I<sub>max</sub>, per-circuit and total
thermal rating, the derated OPF limit, length, corridor R / X / |Z| / X:R, charging Mvar, and the
raw per-km per-circuit r, x, c it was all derived from. The formulas mirror `data_preparation.jl`
— per-length values are *per circuit*, so N parallel circuits divide R and X and multiply the
shunt capacitance — which means the map and the OPF describe the same network.

### Buses — resize by any capacity or demand attribute
Generation capacity (total, or solar / wind / hydro alone), storage power, share of national
demand, installed shunt reactor, line charging at the bus, or generator reactive capability.
Circle area is proportional to the value; grey dots are zero.

A bus popup gives everything at once: base kV, country, NUTS-3 region; demand as both a share of
national load and MW at `REF_LOAD_MW`; the generation fleet broken down by fuel with the hydro
technology split (reservoir / run-of-river / pumped); storage as pumped hydro plus the scenario's
battery fleet in MW and MWh; and the reactive picture — line charging in, shunt reactor out, the
net, and the generators' own ± capability.

**Reactive capability is derived, not measured.** It applies the capability model from
`config.toml [generators]` to installed capacity: synchronous units get
`Q = √(S² − P²)` with `S = P / rated_power_factor`, inverter-based wind and solar get
`Q = tan(acos(renewable_power_factor)) · P`. Both power factors are read from the config, so the
numbers track whatever the model is currently set to.

### Transformers
Their own toggleable layer (off by default). Both terminals of every transformer are colocated,
so each is drawn as one marker; the popup carries the ratio, HV side, unit count, per-unit and
installed MVA, and reactance in pu and in ohms referred to each side.

### Scenario
`SCENARIO` selects only the **battery fleet** — the nodal disaggregation written by
`empire_nodal.jl` to `results/<label>/nodal/bess_units.csv`. Everything else on the map is the
2024 base system and is identical across scenarios. Pick `'2024'` for no batteries; a scenario
that hasn't been disaggregated yet simply shows zero BESS and says so.


In [ ]:
import json, math, webbrowser
from pathlib import Path
import pandas as pd
import numpy as np
import folium

PROJECT = Path.cwd()
OUT = PROJECT / 'results' / 'grid_maps'; OUT.mkdir(parents=True, exist_ok=True)

# ---- Select the scenario once for the whole notebook ----
# The *topology* below is the 2024 network and is identical for every scenario.
# The scenario only adds its nodal BESS fleet, when results/<label>/nodal/ exists
# (written by empire_nodal.jl). '2024' = the base system, no BESS layer.
SCENARIOS = ('2024', 'GoRES', 'NECPEssentials', 'REPowerEU++', 'Trinity')
SCENARIO = 'GoRES'
if SCENARIO not in SCENARIOS:
    raise ValueError(f'SCENARIO must be one of {SCENARIOS}; got {SCENARIO!r}')

def results_dir(scenario):
    return PROJECT / 'results' if scenario in (None, '2024') else PROJECT / 'results' / scenario

# ---- Constants that mirror the Julia model, so the map and the OPF agree ----
FREQ_HZ = 50.0
# National demand used to turn each bus's load SHARE into MW.  data_preparation.jl
# uses TOTAL_LOAD_MW = 35 000 for the 2024 base case; change it to read the map in
# the units you care about (the share itself is scenario-independent).
REF_LOAD_MW = 35_000.0

def _read_cfg(path='config.toml'):
    try:
        import tomllib; loader = tomllib
    except ModuleNotFoundError:
        import tomli as loader
    with open(path, 'rb') as f:
        return loader.load(f)

CFG = _read_cfg()
_net, _gen = CFG.get('network', {}), CFG.get('generators', {})
LRF        = float(_net.get('line_rating_factor', 0.8))     # OPF derate on thermal ratings
PF_SYNC    = float(_gen.get('rated_power_factor', 0.90))    # thermal/hydro MVA circle
PF_INV     = float(_gen.get('renewable_power_factor', 0.95))  # wind/solar inverter cap
# Reactive headroom per MW of installed capacity, from the capability model in
# data_preparation.jl:  synchronous  Q = sqrt(S^2 - P^2) with S = P / pf;
#                       inverter     Q = tan(acos(pf)) * P.
QK_SYNC = math.sqrt(1.0 / PF_SYNC**2 - 1.0)
QK_INV  = math.tan(math.acos(PF_INV))
INVERTER_FUELS = {'Solar', 'Wind'}

# ---- Static system data ----
# Bus_Data.csv carries a UTF-8 BOM on the first header -> utf-8-sig.
buses  = pd.read_csv('Data/Bus_Data.csv', encoding='utf-8-sig')
lines  = pd.read_csv('Data/lines.csv')
xfmr   = pd.read_csv('Data/transformers_reactance.csv')
gens   = pd.read_csv('Data/generations.csv')
loadsh = pd.read_csv('Data/load.csv')
react  = pd.read_csv('Data/reactors.csv')
nuts3  = pd.read_csv('Data/bus_nuts3.csv')
storage_zonal = pd.read_csv('Data/Storage.csv')

BUS = buses.set_index('bus_id')
lines = lines[lines['bus0'].isin(BUS.index) & lines['bus1'].isin(BUS.index)].copy()
xfmr  = xfmr[xfmr['bus0'].isin(BUS.index) & xfmr['bus1'].isin(BUS.index)].copy()

# scenario BESS (nodal disaggregation), if that scenario has been disaggregated
_bess_f = results_dir(SCENARIO) / 'nodal' / 'bess_units.csv'
bess = pd.read_csv(_bess_f) if _bess_f.exists() else pd.DataFrame(
    columns=['bus_id', 'power_mw', 'energy_mwh'])

VOLT_COLOR = {400: '#1f77b4', 320: '#9467bd', 250: '#8c564b',
              225: "#d62728", 220: '#d62728', 132: '#2ca02c'}

print(f'scenario           : {SCENARIO}')
print(f'buses / lines / trf: {len(BUS)} / {len(lines)} / {len(xfmr)}')
print(f'generating units   : {len(gens)}  ({gens.capacity_mw.sum()/1000:.1f} GW)')
print(f'line voltages [kV] : {sorted(lines.voltage.unique())}')
print(f'nodal BESS         : ' + (f'{len(bess)} buses, {bess.power_mw.sum()/1000:.2f} GW / '
                                  f'{bess.energy_mwh.sum()/1000:.1f} GWh'
                                  if len(bess) else 'none (run empire_nodal.jl for this scenario)'))
print(f'reactive model     : pf_sync={PF_SYNC} -> Q/P={QK_SYNC:.3f}, '
      f'pf_inverter={PF_INV} -> Q/P={QK_INV:.3f}; line_rating_factor={LRF:g}')

scenario           : GoRES
buses / lines / trf: 1236 / 2175 / 170
generating units   : 973  (123.1 GW)
line voltages [kV] : [132, 220, 225, 250, 320, 400]
nodal BESS         : 417 buses, 10.27 GW / 20.5 GWh
reactive model     : pf_sync=0.9 -> Q/P=0.484, pf_inverter=0.95 -> Q/P=0.329; line_rating_factor=0.8


In [11]:
# ============================================================================
# Derive every static attribute the map shows, once, into two tables:
#   LN  — one row per transmission line   (geometry, ratings, impedance)
#   BS  — one row per bus                 (fleet, storage, demand, reactive)
# The electrical formulas mirror data_preparation.jl so the map and the OPF
# describe the same network.
# ============================================================================

# ---------------------------------------------------------------- lines ----
LN = lines.copy()
LN['circuits'] = pd.to_numeric(LN['circuits'], errors='coerce').fillna(1.0)
LN['is_dc']    = LN['dc'].astype(str).str.lower().isin(['t', 'true', '1'])

# Thermal rating: S = sqrt(3) * V[kV] * Imax[kA] per circuit, times the circuits.
LN['cap_per_circuit_mw'] = np.sqrt(3) * LN['voltage'] * LN['Imax']
LN['cap_mw']             = LN['cap_per_circuit_mw'] * LN['circuits']
LN['opf_limit_mw']       = LN['cap_mw'] * LRF

# Series impedance of the whole corridor: per-length values are PER CIRCUIT, so
# N parallel circuits divide R and X; shunt capacitance multiplies instead.
LN['R_ohm'] = LN['r_per_length'] * LN['length'] / LN['circuits']
LN['X_ohm'] = LN['x_per_length'] * LN['length'] / LN['circuits']
LN['Z_ohm'] = np.hypot(LN['R_ohm'], LN['X_ohm'])
LN['XR']    = np.where(LN['R_ohm'] > 0, LN['X_ohm'] / LN['R_ohm'], np.nan)
# Charging reactive power at nominal voltage:  Q = 2*pi*f*C*V^2  (kV & F -> Mvar)
_C_farad = LN['c_per_length'] * 1e-9 * LN['length'] * LN['circuits']
LN['Qc_mvar'] = 2 * np.pi * FREQ_HZ * _C_farad * LN['voltage'] ** 2

# --------------------------------------------------------- transformers ----
TR = xfmr.copy()
TR['n_units'] = pd.to_numeric(TR['n_recommended'], errors='coerce').fillna(1.0)

# ----------------------------------------------------------------- buses ----
BS = buses.copy()
BS = BS.merge(nuts3[['bus_id', 'nuts3']], on='bus_id', how='left')

# Generation fleet: totals, per fuel, and the hydro/storage technology split.
g = gens[gens['bus_id'].isin(BS['bus_id'])]
_by_fuel = g.pivot_table(index='bus_id', columns='primary_fuel', values='capacity_mw',
                         aggfunc='sum', fill_value=0.0)
FUELS = ['Nuclear', 'Gas', 'Coal', 'Oil', 'Biomass', 'Hydro', 'Wind', 'Solar']
for f in FUELS:
    BS[f'cap_{f}'] = BS['bus_id'].map(_by_fuel[f] if f in _by_fuel else pd.Series(dtype=float)).fillna(0.0)
BS['cap_total'] = BS[[f'cap_{f}' for f in FUELS]].sum(axis=1)
BS['n_units']   = BS['bus_id'].map(g.groupby('bus_id').size()).fillna(0).astype(int)

_by_tech = g.pivot_table(index='bus_id', columns='technology', values='capacity_mw',
                         aggfunc='sum', fill_value=0.0)
def _tech(name):
    return BS['bus_id'].map(_by_tech[name] if name in _by_tech else pd.Series(dtype=float)).fillna(0.0)
BS['hydro_reservoir_mw'] = _tech('reservoir')
BS['hydro_ror_mw']       = _tech('run_of_river')
BS['pumped_mw']          = _tech('pumped_storage')

# Scenario battery fleet (empty for '2024' or an undisaggregated scenario).
_b = bess.groupby('bus_id')[['power_mw', 'energy_mwh']].sum() if len(bess) else None
BS['bess_mw']  = BS['bus_id'].map(_b['power_mw']).fillna(0.0)  if _b is not None else 0.0
BS['bess_mwh'] = BS['bus_id'].map(_b['energy_mwh']).fillna(0.0) if _b is not None else 0.0
BS['storage_mw'] = BS['pumped_mw'] + BS['bess_mw']

# Demand: load.csv holds an unnormalised share; run_opf.jl uses
# pd_bus = demand / sum(demand) * total_load_mw, so the share is what is portable.
_sh = BS['bus_id'].map(loadsh.set_index('bus_id')['demand']).fillna(0.0)
BS['load_share'] = _sh / _sh.sum()
BS['load_pct']   = BS['load_share'] * 100.0
BS['load_mw']    = BS['load_share'] * REF_LOAD_MW

# Reactive: installed shunt reactors and the line charging seen at each bus
# (both straight from reactors.csv, which sizes the reactors against charging).
_r = react.set_index('bus_id')
BS['reactor_mvar']  = BS['bus_id'].map(_r['reactor_installed_mvar']).fillna(0.0)
BS['charging_mvar'] = BS['bus_id'].map(_r['connected_charging_mvar']).fillna(0.0)
BS['net_shunt_mvar'] = BS['charging_mvar'] - BS['reactor_mvar']   # >0 = net capacitive

# Generator reactive capability at full output, per the capability model above.
_qk = g['primary_fuel'].map(lambda f: QK_INV if f in INVERTER_FUELS else QK_SYNC)
BS['qcap_mvar'] = BS['bus_id'].map(
    (g['capacity_mw'] * _qk).groupby(g['bus_id']).sum()).fillna(0.0)

# ------------------------------------------------- generating units (search) ----
# Kept as its own table so the map's search box can resolve a unit_id or a plant
# name back to the bus it sits on.
GU = gens[gens['bus_id'].isin(BS['bus_id'])].copy()
GU['name'] = GU['name'].fillna('').astype(str)

print(f'lines : cap {LN.cap_mw.min():.0f}-{LN.cap_mw.max():.0f} MW | '
      f'X {LN.X_ohm.min():.2f}-{LN.X_ohm.max():.1f} ohm | '
      f'charging {LN.Qc_mvar.sum():.0f} Mvar total | {int(LN.is_dc.sum())} DC')
print(f'buses : {int((BS.cap_total > 0).sum())} with generation, '
      f'{int((BS.storage_mw > 0).sum())} with storage, '
      f'{int((BS.reactor_mvar > 0).sum())} with a shunt reactor')
print(f'fleet : {BS.cap_total.sum()/1000:.1f} GW  '
      f'(solar {BS.cap_Solar.sum()/1000:.1f}, wind {BS.cap_Wind.sum()/1000:.1f}, '
      f'hydro {BS.cap_Hydro.sum()/1000:.1f}, gas {BS.cap_Gas.sum()/1000:.1f} GW)')
print(f'storage: pumped {BS.pumped_mw.sum()/1000:.2f} GW + '
      f'BESS {BS.bess_mw.sum()/1000:.2f} GW / {BS.bess_mwh.sum()/1000:.1f} GWh')
print(f'reactive: reactors {BS.reactor_mvar.sum()/1000:.1f} Gvar vs charging '
      f'{BS.charging_mvar.sum()/1000:.1f} Gvar; generator capability '
      f'{BS.qcap_mvar.sum()/1000:.1f} Gvar')
print(f'demand : share sums to {BS.load_share.sum():.4f}; '
      f'largest bus {BS.load_pct.max():.2f} % of national demand')
print(f'search : {len(BS)} buses + {len(LN)} lines + {len(TR)} transformers + '
      f'{len(GU)} units indexed')

lines : cap 169-7150 MW | X 0.00-56.1 ohm | charging 36077 Mvar total | 1 DC
buses : 614 with generation, 431 with storage, 1236 with a shunt reactor
fleet : 123.1 GW  (solar 33.2, wind 30.4, hydro 20.5, gas 28.0 GW)
storage: pumped 8.18 GW + BESS 10.27 GW / 20.5 GWh
reactive: reactors 15.5 Gvar vs charging 21.3 Gvar; generator capability 49.7 Gvar
demand : share sums to 1.0000; largest bus 2.30 % of national demand
search : 1236 buses + 2175 lines + 170 transformers + 973 units indexed


In [12]:
# ============================================================================
# THE map.  One self-contained HTML file carrying every static attribute of the
# system; the control panel (top right) only restyles layers that are already
# there, so nothing is recomputed and nothing is re-downloaded.
#
#   lines         — recolour by voltage / capacity / reactance / X:R / circuits
#                   / length / charging;  width always follows thermal capacity
#   buses         — resize by generation, solar, wind, hydro, storage, demand
#                   share, shunt reactor, line charging or generator Q capability
#   transformers  — their own togglable layer
#   popups        — the full record for whatever you click
# ============================================================================

# Rendering ~3 MB inline would bake it into the .ipynb on every save.
SHOW_INLINE = False

BUS_POS = {b: i for i, b in enumerate(BS['bus_id'])}

# ---- line colouring modes: (key, label, unit, ramp kind) -------------------
LINE_MODES = [('voltage',  'voltage level',    'kV',   'discrete'),
              ('cap',      'thermal capacity', 'MW',   'cont'),
              ('X',        'series reactance', 'ohm',  'cont'),
              ('XR',       'X : R ratio',      '',     'cont'),
              ('circuits', 'circuits',         '',     'cont'),
              ('length',   'length',           'km',   'cont'),
              ('Qc',       'charging',         'Mvar', 'cont')]
LINE_VALS = {'voltage': LN['voltage'], 'cap': LN['cap_mw'], 'X': LN['X_ohm'],
             'XR': LN['XR'], 'circuits': LN['circuits'], 'length': LN['length'],
             'Qc': LN['Qc_mvar']}

# ---- bus sizing modes: (column, label, unit, colour) -----------------------
BUS_MODES = [('none',           'no bus overlay',                '',     '#888888'),
             ('cap_total',      'generation capacity',           'MW',   '#2e7d32'),
             ('cap_Solar',      'solar capacity',                'MW',   '#f9a825'),
             ('cap_Wind',       'wind capacity',                 'MW',   '#43a047'),
             ('cap_Hydro',      'hydro capacity',                'MW',   '#1e88e5'),
             ('storage_mw',     'storage power (pumped + BESS)', 'MW',   '#00acc1'),
             ('load_pct',       'share of national demand',      '%',    '#ef6c00'),
             ('reactor_mvar',   'installed shunt reactor',       'Mvar', '#7b1fa2'),
             ('charging_mvar',  'line charging at the bus',      'Mvar', '#00838f'),
             ('qcap_mvar',      'generator reactive capability', 'Mvar', '#c2185b')]

CONT_RAMP = ['#440154', '#3b528b', '#21918c', '#5ec962', '#fde725']   # viridis

def _domain(s):
    """2nd-98th percentile, so a handful of outliers do not flatten the scale."""
    v = pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
    if v.empty:
        return [0.0, 1.0]
    lo, hi = float(np.percentile(v, 2)), float(np.percentile(v, 98))
    return [lo, hi if hi > lo else lo + 1.0]

def _num(s, nd=3):
    a = pd.to_numeric(s, errors='coerce').to_numpy(float)
    return [None if not np.isfinite(x) else round(float(x), nd) for x in a]

payload = dict(
    scenario=SCENARIO, lrf=LRF, refLoad=REF_LOAD_MW,
    pfSync=PF_SYNC, pfInv=PF_INV,
    contRamp=CONT_RAMP,
    lineModes=[[k, lbl, u, kind] for k, lbl, u, kind in LINE_MODES],
    lineDomain={k: _domain(LINE_VALS[k]) for k, _, _, kind in LINE_MODES if kind == 'cont'},
    voltColor={str(int(v)): VOLT_COLOR.get(int(v), '#7f7f7f')
               for v in sorted(LN['voltage'].unique())},
    busModes=[[c, lbl, u, col] for c, lbl, u, col in BUS_MODES],

    # ---- buses ----
    busLL=[[round(float(y), 5), round(float(x), 5)] for y, x in zip(BS['y'], BS['x'])],
    busId=BS['bus_id'].tolist(),
    busNuts=BS['nuts3'].fillna('?').tolist(),
    busCountry=BS['country'].fillna('?').tolist(),
    bus={c: _num(BS[c], 3) for c in
         ['voltage', 'cap_total', 'n_units', 'hydro_reservoir_mw', 'hydro_ror_mw',
          'pumped_mw', 'bess_mw', 'bess_mwh', 'storage_mw', 'load_pct', 'load_mw',
          'reactor_mvar', 'charging_mvar', 'net_shunt_mvar', 'qcap_mvar']
         + [f'cap_{f}' for f in FUELS]},

    # ---- lines ----
    lineA=[BUS_POS[b] for b in LN['bus0']],
    lineB=[BUS_POS[b] for b in LN['bus1']],
    lineId=LN['line_id'].tolist(),
    lineDc=[int(b) for b in LN['is_dc']],
    line={k: _num(LINE_VALS[k], 4) for k, _, _, _ in LINE_MODES}
         | {c: _num(LN[c], 4) for c in
            ['Imax', 'cap_per_circuit_mw', 'cap_mw', 'opf_limit_mw',
             'R_ohm', 'X_ohm', 'Z_ohm', 'Qc_mvar', 'r_per_length',
             'x_per_length', 'c_per_length']},

    # ---- transformers (both terminals are colocated, so draw one marker) ----
    trLL=[[round(float(BS['y'][BUS_POS[b]]), 5), round(float(BS['x'][BUS_POS[b]]), 5)]
          for b in TR['bus0']],
    trId=TR['transformer_id'].tolist(),
    trBus=[[a, b] for a, b in zip(TR['bus0'], TR['bus1'])],
    tr={c: _num(TR[c], 4) for c in
        ['voltage_bus0', 'voltage_bus1', 'HV_kV', 'n_units', 'per_unit_MVA',
         'installed_MVA', 'X_pu_on_installed_base', 'X_ohm_HV', 'X_ohm_LV']},

    # ---- generating units: searchable, and resolved to the bus they sit on ----
    unitId=GU['unit_id'].astype(str).tolist(),
    unitName=GU['name'].tolist(),
    unitBus=[BUS_POS[b] for b in GU['bus_id']],
    unitFuel=GU['primary_fuel'].astype(str).tolist(),
    unitTech=GU['technology'].astype(str).tolist(),
    unitCap=_num(GU['capacity_mw'], 1),
)
blob = json.dumps(payload, separators=(',', ':'))

m = folium.Map(location=[40.0, -3.6], zoom_start=6, tiles='cartodbpositron', control_scale=True)

# NUTS-3 province borders (cached locally after the first download)
try:
    import urllib.request
    _cache = Path('Data/nuts3_es.geojson')
    if _cache.exists():
        _fc = json.loads(_cache.read_text(encoding='utf-8'))
    else:
        _u = ('https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/'
              'NUTS_RG_20M_2021_4326_LEVL_3.geojson')
        with urllib.request.urlopen(_u, timeout=30) as _r:
            _n = json.load(_r)
        _fc = {'type': 'FeatureCollection',
               'features': [f for f in _n['features'] if f['properties']['CNTR_CODE'] == 'ES']}
        _cache.write_text(json.dumps(_fc), encoding='utf-8')
    _fg = folium.FeatureGroup(name='NUTS-3 borders', show=False)
    folium.GeoJson(_fc, style_function=lambda _f: dict(color='#888', weight=0.8, fill=False)).add_to(_fg)
    _fg.add_to(m)
except Exception as _e:
    print('NUTS-3 layer skipped:', _e)

CSS = """
<style>
.gridctl{background:#fff;padding:9px 11px;border-radius:6px;box-shadow:0 1px 5px rgba(0,0,0,.4);
  font:12px/1.35 -apple-system,Segoe UI,Helvetica,Arial,sans-serif;width:238px;
  max-height:86vh;overflow-y:auto}
.gridctl label{display:block;cursor:pointer;padding:1px 0}
.gridctl hr{border:0;border-top:1px solid #ddd;margin:7px 0}
.gridctl .hint{color:#777;font-size:10.5px;display:block;margin-top:3px}
.gridctl .bar{height:9px;border-radius:2px;margin:4px 0 2px}
.gridctl .ends{display:flex;justify-content:space-between;color:#555;font-size:10.5px}
.gridpop{font:12px/1.45 -apple-system,Segoe UI,Helvetica,Arial,sans-serif;max-height:340px;overflow-y:auto}
.gridpop h4{margin:6px 0 2px;font-size:12px;color:#1565c0}
.gridpop table{border-collapse:collapse}
.gridpop td{padding:0 8px 0 0;vertical-align:top}
.gridpop td.v{font-weight:600;text-align:right}
.gridctl input[type=text]{width:100%;box-sizing:border-box;padding:4px 6px;margin:4px 0;
  border:1px solid #bbb;border-radius:3px;font:12px inherit}
.qrow{padding:3px 5px;cursor:pointer;border-radius:3px;display:flex;gap:6px;align-items:baseline}
.qrow:hover{background:#eef5fb}
.qrow .tag{font-size:9.5px;text-transform:uppercase;letter-spacing:.03em;color:#fff;
  padding:1px 4px;border-radius:2px;flex:none}
.qrow .nm{overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.qrow .sub{color:#888;font-size:10.5px;margin-left:auto;text-align:right;flex:none}
#qList{max-height:190px;overflow-y:auto;margin-top:2px}
</style>
"""
m.get_root().header.add_child(folium.Element(CSS))

JS = r"""
// folium emits this into <head>, before the map exists, so defer until load.
window.addEventListener('load', function(){
var MAP = __MAP__;
var P = __DATA__;
var lineMode = 'voltage', busMode = 'none';

function hex2rgb(h){return [parseInt(h.substr(1,2),16),parseInt(h.substr(3,2),16),parseInt(h.substr(5,2),16)];}
function lerpRamp(ramp, t){
  t = Math.max(0, Math.min(1, t)) * (ramp.length - 1);
  var i = Math.min(ramp.length - 2, Math.floor(t)), f = t - i, a = ramp[i], b = ramp[i+1];
  return 'rgb(' + Math.round(a[0]+(b[0]-a[0])*f) + ',' + Math.round(a[1]+(b[1]-a[1])*f)
       + ',' + Math.round(a[2]+(b[2]-a[2])*f) + ')';
}
var CRAMP = P.contRamp.map(hex2rgb);
function fmt(v, d){
  if (v === null || v === undefined || !isFinite(v)) return 'n/a';
  // group the integer part only — grouping the fraction would render 0.03 as "0.0 30"
  var s = v.toFixed(d === undefined ? 0 : d).split('.');
  s[0] = s[0].replace(/\B(?=(\d{3})+(?!\d))/g, ' ');
  return s.join('.');
}
function modeInfo(list, key){
  for (var i = 0; i < list.length; i++) if (list[i][0] === key) return list[i];
  return null;
}
function row(k, v){ return '<tr><td>' + k + '</td><td class="v">' + v + '</td></tr>'; }

// ------------------------------------------------------------------ lines --
function lineColor(i){
  if (lineMode === 'voltage')
    return P.voltColor[String(P.line.voltage[i])] || '#7f7f7f';
  var d = P.lineDomain[lineMode], v = P.line[lineMode][i];
  if (v === null || v === undefined) return '#bbbbbb';
  return lerpRamp(CRAMP, (v - d[0]) / (d[1] - d[0]));
}
var CAPS = P.line.cap_mw.filter(function(v){ return v !== null; });
var CAPMIN = Math.min.apply(null, CAPS), CAPMAX = Math.max.apply(null, CAPS);
function lineWidth(i){
  var c = P.line.cap_mw[i];
  return c === null ? 0.8 : 0.8 + 5.2 * (c - CAPMIN) / Math.max(CAPMAX - CAPMIN, 1e-9);
}
function linePopup(i){
  var L = P.line;
  return '<div class="gridpop"><b>' + P.lineId[i] + '</b>'
    + (P.lineDc[i] ? ' <span style="color:#7b1fa2">(DC link)</span>' : '')
    + '<br><span style="color:#777">' + P.busId[P.lineA[i]] + ' &harr; ' + P.busId[P.lineB[i]]
    + '</span><h4>Ratings</h4><table>'
    + row('voltage', fmt(L.voltage[i]) + ' kV')
    + row('circuits', fmt(L.circuits[i]))
    + row('I<sub>max</sub>', fmt(L.Imax[i], 2) + ' kA')
    + row('rating / circuit', fmt(L.cap_per_circuit_mw[i]) + ' MW')
    + row('<b>total capacity</b>', '<b>' + fmt(L.cap_mw[i]) + ' MW</b>')
    + row('OPF limit (&times;' + P.lrf + ')', fmt(L.opf_limit_mw[i]) + ' MW')
    + '</table><h4>Geometry &amp; impedance</h4><table>'
    + row('length', fmt(L.length[i], 1) + ' km')
    + row('R (corridor)', fmt(L.R_ohm[i], 3) + ' &#8486;')
    + row('X (corridor)', fmt(L.X_ohm[i], 3) + ' &#8486;')
    + row('|Z|', fmt(L.Z_ohm[i], 3) + ' &#8486;')
    + row('X : R', fmt(L.XR[i], 2))
    + row('charging', fmt(L.Qc_mvar[i], 2) + ' Mvar')
    + '</table><span class="hint">per km &amp; per circuit: r ' + fmt(L.r_per_length[i], 4)
    + ', x ' + fmt(L.x_per_length[i], 4) + ' &#8486;/km, c ' + fmt(L.c_per_length[i], 1)
    + ' nF/km</span></div>';
}
var lineGroup = L.layerGroup().addTo(MAP), lineLayers = [];
P.lineA.forEach(function(a, i){
  var pl = L.polyline([P.busLL[a], P.busLL[P.lineB[i]]],
                      {color:'#999', weight:lineWidth(i), opacity:0.85,
                       dashArray: P.lineDc[i] ? '5,6' : null});
  pl.bindTooltip(function(){
    var mi = modeInfo(P.lineModes, lineMode);
    return '<b>' + P.lineId[i] + '</b><br>' + fmt(P.line.voltage[i]) + ' kV, '
         + fmt(P.line.cap_mw[i]) + ' MW<br>' + mi[1] + ': '
         + fmt(P.line[lineMode][i], 2) + ' ' + mi[2];
  }, {sticky:true});
  pl.on('click', function(e){ pl.bindPopup(linePopup(i), {maxWidth:360}).openPopup(e.latlng); });
  pl.addTo(lineGroup);
  lineLayers.push(pl);
});

// ----------------------------------------------------------- transformers --
var trGroup = L.layerGroup();
function trPopup(i){
  var T = P.tr;
  return '<div class="gridpop"><b>' + P.trId[i] + '</b><br>'
    + '<span style="color:#777">' + P.trBus[i][0] + ' &harr; ' + P.trBus[i][1]
    + '</span><h4>Transformer</h4><table>'
    + row('ratio', fmt(T.voltage_bus0[i]) + ' / ' + fmt(T.voltage_bus1[i]) + ' kV')
    + row('HV side', fmt(T.HV_kV[i]) + ' kV')
    + row('units', fmt(T.n_units[i]))
    + row('rating / unit', fmt(T.per_unit_MVA[i]) + ' MVA')
    + row('<b>installed</b>', '<b>' + fmt(T.installed_MVA[i]) + ' MVA</b>')
    + row('X (on installed base)', fmt(T.X_pu_on_installed_base[i], 4) + ' pu')
    + row('X (HV / LV)', fmt(T.X_ohm_HV[i], 2) + ' / ' + fmt(T.X_ohm_LV[i], 2) + ' &#8486;')
    + '</table></div>';
}
P.trLL.forEach(function(ll, i){
  var T = P.tr;
  var mk = L.circleMarker(ll, {radius:3.4, color:'#5d4037', weight:1.2,
                               fillColor:'#a1887f', fillOpacity:0.9});
  mk.bindTooltip('<b>' + P.trId[i] + '</b><br>' + fmt(T.voltage_bus0[i]) + '/'
               + fmt(T.voltage_bus1[i]) + ' kV, ' + fmt(T.installed_MVA[i]) + ' MVA',
               {sticky:true});
  mk.bindPopup(trPopup(i), {maxWidth:340});
  mk.addTo(trGroup);
});

// ------------------------------------------------------------------ buses --
var B = P.bus;
var BUSMAX = {};
P.busModes.forEach(function(mm){
  if (mm[0] === 'none') return;
  var mx = 0, arr = B[mm[0]];
  for (var j = 0; j < arr.length; j++) if (arr[j] > mx) mx = arr[j];
  BUSMAX[mm[0]] = mx || 1;
});
function busPopup(j){
  var fuels = ['Nuclear','Gas','Coal','Oil','Biomass','Hydro','Wind','Solar'];
  var mix = '';
  fuels.forEach(function(f){
    var v = B['cap_' + f][j];
    if (v > 0.5) mix += row(f, fmt(v) + ' MW');
  });
  var h = '<div class="gridpop"><b>' + P.busId[j] + '</b> | ' + fmt(B.voltage[j])
    + ' kV | ' + P.busCountry[j] + ' | NUTS-3 ' + P.busNuts[j]
    + '<h4>Demand</h4><table>'
    + row('share of national', fmt(B.load_pct[j], 3) + ' %')
    + row('at ' + fmt(P.refLoad) + ' MW national', fmt(B.load_mw[j], 1) + ' MW')
    + '</table>';
  h += '<h4>Generation (' + fmt(B.n_units[j]) + ' units, '
     + fmt(B.cap_total[j]) + ' MW)</h4>';
  h += mix ? '<table>' + mix + '</table>' : '<span class="hint">no units at this bus</span>';
  if (B.hydro_reservoir_mw[j] + B.hydro_ror_mw[j] + B.pumped_mw[j] > 0.5)
    h += '<table>' + row('&nbsp;&nbsp;hydro: reservoir', fmt(B.hydro_reservoir_mw[j]) + ' MW')
       + row('&nbsp;&nbsp;hydro: run-of-river', fmt(B.hydro_ror_mw[j]) + ' MW')
       + row('&nbsp;&nbsp;hydro: pumped', fmt(B.pumped_mw[j]) + ' MW') + '</table>';
  h += '<h4>Storage</h4><table>'
     + row('pumped hydro', fmt(B.pumped_mw[j]) + ' MW')
     + row('battery (' + P.scenario + ')', fmt(B.bess_mw[j]) + ' MW / '
           + fmt(B.bess_mwh[j]) + ' MWh')
     + row('<b>total power</b>', '<b>' + fmt(B.storage_mw[j]) + ' MW</b>')
     + '</table><h4>Reactive power</h4><table>'
     + row('line charging in', fmt(B.charging_mvar[j], 1) + ' Mvar')
     + row('shunt reactor', '&minus;' + fmt(B.reactor_mvar[j], 1) + ' Mvar')
     + row('net shunt', fmt(B.net_shunt_mvar[j], 1) + ' Mvar')
     + row('generator capability', '&plusmn;' + fmt(B.qcap_mvar[j], 1) + ' Mvar')
     + '</table><span class="hint">capability at full output, pf ' + P.pfSync
     + ' synchronous / ' + P.pfInv + ' inverter</span></div>';
  return h;
}
var busGroup = L.layerGroup().addTo(MAP), busLayers = [];
P.busLL.forEach(function(ll, j){
  var cm = L.circleMarker(ll, {radius:1.6, weight:0.5, color:'#555', opacity:0.7,
                               fillColor:'#888', fillOpacity:0.7});
  cm.bindTooltip(function(){
    if (busMode === 'none')
      return '<b>' + P.busId[j] + '</b><br>' + fmt(B.voltage[j]) + ' kV';
    var mi = modeInfo(P.busModes, busMode);
    return '<b>' + P.busId[j] + '</b><br>' + mi[1] + ': '
         + fmt(B[busMode][j], busMode === 'load_pct' ? 3 : 1) + ' ' + mi[2];
  }, {sticky:true});
  cm.on('click', function(e){ cm.bindPopup(busPopup(j), {maxWidth:330}).openPopup(e.latlng); });
  cm.addTo(busGroup);
  busLayers.push(cm);
});

// ----------------------------------------------------------------- search --
// Halos live in their own pane BELOW the overlay pane, so a highlighted line keeps
// showing its own colour instead of being painted over.
MAP.createPane('halo');
MAP.getPane('halo').style.zIndex = 399;          // overlayPane is 400
MAP.getPane('halo').style.pointerEvents = 'none';
var haloGroup = L.layerGroup().addTo(MAP);
var HALO = '#e5007d', MAX_HALO = 400;

function unitPopup(u){
  return '<div class="gridpop"><b>' + P.unitId[u] + '</b>'
    + (P.unitName[u] ? '<br><span style="color:#777">' + P.unitName[u] + '</span>' : '')
    + '<h4>Generating unit</h4><table>'
    + row('fuel', P.unitFuel[u])
    + row('technology', P.unitTech[u])
    + row('<b>capacity</b>', '<b>' + fmt(P.unitCap[u]) + ' MW</b>')
    + row('at bus', P.busId[P.unitBus[u]])
    + '</table><span class="hint">Click the bus itself for its full record.</span></div>';
}

// one flat index over everything that has an id
var IDX = [], TAGCOL = {bus:'#1565c0', line:'#2e7d32', trafo:'#5d4037', unit:'#c2185b'};
P.busId.forEach(function(id, j){
  IDX.push({t:'bus', id:id, j:j, sub:fmt(B.voltage[j]) + ' kV'}); });
P.lineId.forEach(function(id, i){
  IDX.push({t:'line', id:id, i:i,
            sub:fmt(P.line.voltage[i]) + ' kV / ' + fmt(P.line.cap_mw[i]) + ' MW'}); });
P.trId.forEach(function(id, i){
  IDX.push({t:'trafo', id:id, i:i, sub:fmt(P.tr.installed_MVA[i]) + ' MVA'}); });
P.unitId.forEach(function(id, u){
  IDX.push({t:'unit', id:id, u:u, j:P.unitBus[u], name:P.unitName[u],
            sub:P.unitFuel[u] + ' / ' + fmt(P.unitCap[u]) + ' MW'}); });
IDX.forEach(function(e){ e.hay = (e.id + ' ' + (e.name || '')).toLowerCase(); });

function search(q){
  // split on commas/whitespace so a pasted list of ids highlights all of them at once
  var terms = q.toLowerCase().split(/[,;\s]+/).filter(function(s){ return s.length; });
  if (!terms.length) return [];
  var out = [];
  IDX.forEach(function(e){
    for (var k = 0; k < terms.length; k++){
      if (e.hay.indexOf(terms[k]) >= 0){
        e._exact = (e.id.toLowerCase() === terms[k]);
        out.push(e);
        return;
      }
    }
  });
  out.sort(function(a, b){
    if (a._exact !== b._exact) return a._exact ? -1 : 1;
    if (a.t !== b.t) return a.t < b.t ? -1 : 1;
    return a.id < b.id ? -1 : 1;
  });
  return out;
}
function pointsOf(e){
  if (e.t === 'line') return [P.busLL[P.lineA[e.i]], P.busLL[P.lineB[e.i]]];
  return [e.t === 'trafo' ? P.trLL[e.i] : P.busLL[e.j]];
}
function haloFor(e){
  if (e.t === 'line'){
    var p = pointsOf(e);
    return L.polyline(p, {pane:'halo', color:HALO, weight:lineWidth(e.i) + 7,
                          opacity:0.8, lineCap:'round'});
  }
  return L.circleMarker(pointsOf(e)[0], {pane:'halo', radius: e.t === 'trafo' ? 9 : 11,
                                         color:HALO, weight:3.5, opacity:0.95, fill:false});
}
function highlight(list, fit){
  haloGroup.clearLayers();
  var pts = [];
  list.slice(0, MAX_HALO).forEach(function(e){
    haloFor(e).addTo(haloGroup);
    pointsOf(e).forEach(function(p){ pts.push(p); });
  });
  if (fit && pts.length){
    if (pts.length === 1) MAP.setView(pts[0], Math.max(MAP.getZoom(), 11));
    else MAP.fitBounds(L.latLngBounds(pts).pad(0.25));
  }
}
function focusOn(e){
  highlight([e], true);
  var content = e.t === 'line'  ? linePopup(e.i)
              : e.t === 'trafo' ? trPopup(e.i)
              : e.t === 'unit'  ? unitPopup(e.u)
              :                   busPopup(e.j);
  var p = pointsOf(e);
  var ll = e.t === 'line' ? [(p[0][0]+p[1][0])/2, (p[0][1]+p[1][1])/2] : p[0];
  L.popup({maxWidth:360}).setLatLng(ll).setContent(content).openOn(MAP);
}
function runSearch(fit){
  var q = document.getElementById('qBox').value;
  var res = search(q);
  var counts = {bus:0, line:0, trafo:0, unit:0};
  res.forEach(function(e){ counts[e.t]++; });
  document.getElementById('qList').innerHTML = res.slice(0, 25).map(function(e, k){
    return '<div class="qrow" data-k="' + k + '">'
         + '<span class="tag" style="background:' + TAGCOL[e.t] + '">' + e.t + '</span>'
         + '<span class="nm">' + e.id
         + (e.name ? ' <span style="color:#999">' + e.name + '</span>' : '')
         + '</span><span class="sub">' + e.sub + '</span></div>';
  }).join('');
  Array.prototype.forEach.call(document.querySelectorAll('#qList .qrow'), function(el){
    el.addEventListener('click', function(){ focusOn(res[+this.dataset.k]); });
  });
  document.getElementById('qInfo').textContent = !q.trim() ? ''
    : (res.length ? res.length + ' match' + (res.length === 1 ? '' : 'es') + ' — '
        + counts.bus + ' bus, ' + counts.line + ' line, ' + counts.trafo + ' trafo, '
        + counts.unit + ' unit' + (res.length > 25 ? ' (first 25 listed)' : '')
        + (res.length > MAX_HALO ? '; only ' + MAX_HALO + ' highlighted' : '')
      : 'no match');
  highlight(res, fit);
  return res;
}

// ----------------------------------------------------------------- redraw --
function redraw(){
  for (var i = 0; i < lineLayers.length; i++) lineLayers[i].setStyle({color: lineColor(i)});
  var mi = modeInfo(P.busModes, busMode);
  for (var j = 0; j < busLayers.length; j++){
    if (busMode === 'none'){
      busLayers[j].setStyle({radius:1.6, fillColor:'#888', color:'#555',
                             weight:0.5, fillOpacity:0.7});
    } else {
      var v = B[busMode][j] || 0, mx = BUSMAX[busMode];
      var r = v <= 0 ? 0.9 : 2 + 12 * Math.sqrt(v / mx);
      busLayers[j].setStyle({radius:r, fillColor: v > 0 ? mi[3] : '#cccccc',
                             color:'#333', weight: v > 0 ? 0.6 : 0.3, fillOpacity:0.68});
    }
  }
  document.getElementById('lineLegend').innerHTML = lineLegend();
  document.getElementById('busLegend').innerHTML  = busLegend();
}
function lineLegend(){
  var mi = modeInfo(P.lineModes, lineMode);
  if (mi[3] === 'discrete'){
    var s = '';
    Object.keys(P.voltColor).sort(function(a,b){ return b - a; }).forEach(function(kv){
      s += '<span style="color:' + P.voltColor[kv] + '">&#9644;</span> ' + kv + ' kV &nbsp;';
    });
    return '<span class="hint">' + s + '</span>';
  }
  var d = P.lineDomain[lineMode];
  var stops = P.contRamp.join(',');
  return '<div class="bar" style="background:linear-gradient(to right,' + stops + ')"></div>'
       + '<div class="ends"><span>' + fmt(d[0], 2) + '</span><span>' + (mi[2] || mi[1])
       + '</span><span>' + fmt(d[1], 2) + '</span></div>'
       + '<span class="hint">2nd&ndash;98th percentile; width always follows capacity.</span>';
}
function busLegend(){
  if (busMode === 'none')
    return '<span class="hint">Buses drawn as plain dots. Click any for its full record.</span>';
  var mi = modeInfo(P.busModes, busMode);
  return '<span class="hint"><span style="color:' + mi[3] + '">&#9679;</span> area &prop; '
       + mi[1] + '; largest = ' + fmt(BUSMAX[busMode], busMode === 'load_pct' ? 2 : 0)
       + ' ' + mi[2] + '. Grey = zero.</span>';
}

// ---------------------------------------------------------- control panel --
var Ctl = L.Control.extend({
  options:{position:'topright'},
  onAdd:function(){
    var d = L.DomUtil.create('div', 'gridctl');
    d.innerHTML =
      '<b>Search</b>'
      + '<input type="text" id="qBox" autocomplete="off" spellcheck="false"'
      + ' placeholder="bus / line / transformer / unit id">'
      + '<span class="hint">Substring match on ids and plant names. Paste several,'
      + ' comma- or space-separated, to highlight them all. Enter zooms to the set;'
      + ' click a result to zoom to one and open it.</span>'
      + '<div id="qList"></div><span class="hint" id="qInfo"></span>'
      + '<hr><b>Lines &mdash; colour by</b>'
      + P.lineModes.map(function(mm){
          return '<label><input type="radio" name="lineMode" value="' + mm[0] + '"'
               + (mm[0] === lineMode ? ' checked' : '') + '> ' + mm[1]
               + (mm[2] ? ' [' + mm[2] + ']' : '') + '</label>'; }).join('')
      + '<div id="lineLegend"></div>'
      + '<hr><b>Buses &mdash; size by</b>'
      + P.busModes.map(function(mm){
          return '<label><input type="radio" name="busMode" value="' + mm[0] + '"'
               + (mm[0] === busMode ? ' checked' : '') + '> ' + mm[1]
               + (mm[2] ? ' [' + mm[2] + ']' : '') + '</label>'; }).join('')
      + '<div id="busLegend"></div>'
      + '<hr><b>Layers</b>'
      + '<label><input type="checkbox" id="cbLines" checked> transmission lines</label>'
      + '<label><input type="checkbox" id="cbBuses" checked> buses</label>'
      + '<label><input type="checkbox" id="cbTrafo"> transformers ('
      + P.trId.length + ')</label>';
    L.DomEvent.disableClickPropagation(d);
    L.DomEvent.disableScrollPropagation(d);
    return d;
  }
});
MAP.addControl(new Ctl());

Array.prototype.forEach.call(document.getElementsByName('lineMode'), function(r){
  r.addEventListener('change', function(){ lineMode = this.value; MAP.closePopup(); redraw(); });
});
Array.prototype.forEach.call(document.getElementsByName('busMode'), function(r){
  r.addEventListener('change', function(){ busMode = this.value; MAP.closePopup(); redraw(); });
});
function toggle(id, layer){
  document.getElementById(id).addEventListener('change', function(){
    if (this.checked) layer.addTo(MAP); else MAP.removeLayer(layer);
  });
}
toggle('cbLines', lineGroup); toggle('cbBuses', busGroup); toggle('cbTrafo', trGroup);

var qBox = document.getElementById('qBox'), qTimer = null;
qBox.addEventListener('input', function(){
  clearTimeout(qTimer);
  MAP.closePopup();                        // a popup from the previous query is stale
  qTimer = setTimeout(function(){ runSearch(false); }, 140);   // highlight live, don't pan
});
qBox.addEventListener('keydown', function(ev){
  if (ev.key === 'Enter'){ ev.preventDefault(); clearTimeout(qTimer); runSearch(true); }
  if (ev.key === 'Escape'){ this.value = ''; runSearch(false); MAP.closePopup(); }
});
redraw();
});
"""
m.get_root().script.add_child(folium.Element(
    JS.replace('__MAP__', m.get_name()).replace('__DATA__', blob)))
folium.LayerControl(collapsed=False).add_to(m)

out = OUT / f'system_map_{SCENARIO}.html'
m.save(str(out))
print(f'embedded payload : {len(blob)/1e6:.2f} MB')
print(f'saved            : {out}  ({out.stat().st_size/1e6:.2f} MB)')
print(f'drawn            : {len(LN)} lines, {len(BS)} buses, {len(TR)} transformers')
webbrowser.open(out.resolve().as_uri())
m if SHOW_INLINE else None

embedded payload : 0.56 MB
saved            : c:\Users\ehsanno\DataspellProjects\Spanish_Power_System\results\grid_maps\system_map_GoRES.html  (0.68 MB)
drawn            : 2175 lines, 1236 buses, 170 transformers
